# 传输线特性的建模## 目录* [简介](#introduction) * [传播常数](#propagation_constant) * [关于衰减单位的补充](#attenuation_units)* [使用传输线函数建模带负载的有损传输线](#tline_functions) * [输入阻抗、反射系数和驻波比](#tline_impedances) * [电压和电流](#voltages_currents)* [通过级联网络建模带负载的有损传输线](#cascading_networks)* [从输入阻抗确定传播常数](#propagation_constant_from_zin)## 简介 <a class="anchor" id="introduction"></a>在本教程中，我们将使用 `scikit-rf` 处理一些经典的传输线场景，例如计算阻抗、反射系数、驻波比或电压和电流。至少有两种方法可以执行这些计算，一种是使用[传输线函数](#tline_functions)，另一种是[创建和级联网络](#cascading_networks)。让我们考虑一根特性阻抗为 $Z_0 = 75 \Omega$、长度为 $d = 12 m$ 的有损同轴电缆。该同轴电缆的衰减为 0.02 奈珀/米，[速度因子](https://en.wikipedia.org/wiki/Velocity_factor) VF=0.67（这大致对应于[RG-6](https://en.wikipedia.org/wiki/RG-6)同轴电缆）。电缆的负载阻抗为 $Z_L = 150 \Omega$。感兴趣的射频频率为 250 MHz。请注意，在 `scikit-rf` 中，线路长度是从负载开始定义的，即负载处为 $z=0$，传输线输入端为 $z=d$：<img src="transmission_line_properties.svg">首先，让我们进行必要的 Python 导入：

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.constants import c
from skrf.mathFunctions import db_2_np, db_per_100feet_2_db_per_100meter, feet_2_meter, mag_2_db10, np_2_db
from skrf.tlineFunctions import (
    reflection_coefficient_2_propagation_constant,
    voltage_current_propagation,
    zl_2_Gamma0,
    zl_2_Gamma_in,
    zl_2_swr,
    zl_2_total_loss,
    zl_2_zin,
)


In [ ]:
# skrf figure styling
rf.stylely()

以及问题中的常数：

In [ ]:
freq = rf.Frequency(250, npoints=1, unit='MHz')
Z_0 = 75  # Ohm
Z_L = 150  # Ohm
d = 12  # m
VF = 0.67
att = 0.02 # Np/m. Equivalent to 0.1737 dB/m

在进行阻抗和反射系数计算之前，我们需要先定义传输线的特性，特别是其传播常数。

### 传播常数为了获得传输线的射频参数，有必要推导出该线的传播常数。在 `scikit-rf` 中，该线的传播常数 $\gamma$ 定义为 $\gamma=\alpha + j\beta$，其中 $\alpha$ 是衰减（单位为奈珀/米），$\beta=\frac{2\pi}{\lambda}=\frac{\omega}{c}/\mathrm{VF}=\frac{\omega}{c}\sqrt{\epsilon_r}$ 是相位常数。首先，同轴电缆中的波长为 $$\lambda=\frac{c}{f \sqrt{\epsilon_r}}=\frac{c}{f} \mathrm{VF} $$

In [ ]:
lambd = c/freq.f * VF
print('VF=', VF, 'and Wavelength:', lambd, 'm')

由于衰减已经以 Np/m 的形式给出，因此传播常数为：

In [ ]:
alpha = att  # Np/m !
beta = freq.w/c/VF
gamma = alpha + 1j*beta
print('Transmission line propagation constant: gamma = ', gamma, 'rad/m')

如果衰减值是以其他单位给出的，`scikit-rf` 提供了必要的工具来进行单位转换，具体如下所述。

### 间歇：关于衰减单位的讨论

衰减通常以各种单位给出（或预期给出）。`scikit-rf` 提供了方便的函数来处理线路衰减单位。例如，以 Np/m 为单位给出的电缆衰减，可以表示为 dB：

In [ ]:
print('Attenuation dB/m:', np_2_db(att))

因此，以 dB/100m 为单位的衰减为：

In [ ]:
print('Line attenuation in dB/100m:', np_2_db(att)*100)

以 dB/100 英尺为单位，结果为：

In [ ]:
print('Line attenuation in dB/100ft:', np_2_db(att)*100*feet_2_meter())

如果衰减值以英制单位给出，例如 dB/100 英尺，则相应的转换关系将是：

In [ ]:
db_per_100feet_2_db_per_100meter(5.2949)  # to dB/100m

In [ ]:
db_2_np(5.2949)/feet_2_meter(100)  # to Np/m

## 使用传输线函数 <a class="anchor" id="tline_functions"></a>`scikit-rf` 提供了一些方便的函数来处理传输线。 详细信息请参见[传输线函数](../../api/tlineFunctions.rst)文档页面。### 输入阻抗、反射系数和驻波比 <a class="anchor" id="tline_impedances"></a>负载引起的反射系数 $\Gamma_L$ 由 `zl_2_Gamma0()` 函数计算：

In [ ]:
Gamma0 = zl_2_Gamma0(Z_0, Z_L)
print('|Gamma0|=', abs(Gamma0))

以及与其相关的驻波比 (SWR) 可以从 `zl_2_swr()` 函数中获得：

In [ ]:
zl_2_swr(Z_0, Z_L)

信号在具有传播常数 $\gamma$ 的传输线上传播一段距离 $d$（因此，信号的电长度为 $\theta = \gamma d$）后，可以通过 `zl_2_Gamma_in()` 获得线路输入端的反射系数：

In [ ]:
Gamma_in = zl_2_Gamma_in(Z_0, Z_L, theta=gamma*d)
print('|Gamma_in|=', abs(Gamma_in), 'phase=', 180/np.pi*np.angle(Gamma_in))

从 `zl_2_zin()` 函数获得的输入阻抗 $Z_{in}$：

In [ ]:
Z_in = zl_2_zin(Z_0, Z_L, gamma * d)
print('Input impedance Z_in=', Z_in)

与之前一样，线路输入的驻波比为：

In [ ]:
zl_2_swr(Z_0, Z_in)

以 dB 为单位的总线路损耗可以通过 `zl_2_total_loss()` 函数获得：

In [ ]:
mag_2_db10(zl_2_total_loss(Z_0, Z_L, gamma*d))

### 电压和电流

现在假设前一个电路由一个电压源激励，该电压源的电压为 $V=1 V$，并具有源阻抗 $Z_s=100\Omega$：<img src="transmission_line_properties_vi.svg">

In [ ]:
Z_s = 100  # Ohm
V_s = 1  # V

在传输线的输入端，电压是一个分压电路：$$V_{in} = V_s  \frac{Z_{in}}{Z_s + Z_{in}}$$

In [ ]:
V_in = V_s * Z_in / (Z_s + Z_in)
print('Voltage at transmission line input : V_in = ', V_in, ' V')

传输线的输入电流为：$$I_{in} = \frac{V_s}{Z_s + Z_{in}}$$

In [ ]:
I_in = V_s / (Z_s + Z_in)
print('Current at transmission line input : I_in = ', I_in, ' A')

它表示一个功率，公式如下：$$P_{in} = \frac{1}{2} \Re \left[V_{in} I_{in}^* \right]$$

In [ ]:
P_in = 1/2 * np.real(V_in * np.conj(I_in))
print('Input Power : P_in = ', P_in, 'W')

反射功率为：$$P_r = |\Gamma_{in}|^2 P_{in}$$

In [ ]:
P_r = abs(Gamma_in)**2 * P_in
print('Reflected power : P_r = ', P_r, 'W')

对于长度为 $L$ 的传输线，其端电压和电流可以从 ABCD 参数中推导得出：

In [ ]:
V_out, I_out = voltage_current_propagation(V_in, I_in, Z_0,theta= gamma*d)
print('Voltage at load: V_out = ', V_out, 'V')
print('Current at load: I_out = ', I_out, 'A')

请注意，电压和电流都以峰值表示。因此，均方根值为：

In [ ]:
print(abs(V_out)/np.sqrt(2), abs(I_out)/np.sqrt(2))

因此，传递到负载上的功率为：

In [ ]:
P_out = 1/2 * np.real(V_out * np.conj(I_out))
print('Power delivered to the load : P_out = ', P_out, ' W')

以下图表显示了电压和电流随传输线长度的变化（请注意电压和电流传播中 $d$ 的符号：从源端 ($z=d$) 到负载端 ($z=0$) 时，$\theta$ 的方向相反，因此需要进行反转）。

In [ ]:
ds = np.linspace(0, d, num=1001)

thetas = - gamma*ds

v1 = np.full_like(ds, V_in)
i1 = np.full_like(ds, I_in)

v2, i2 = voltage_current_propagation(v1, i1, Z_0, thetas)

In [ ]:
fig, (ax_V, ax_I) = plt.subplots(2, 1, sharex=True)
ax_V.plot(ds, abs(v2), lw=2)
ax_I.plot(ds, abs(i2), lw=2, c='C1')
ax_I.set_xlabel('z [m]')
ax_V.set_ylabel('|V| [V]')
ax_I.set_ylabel('|I| [A]')


ax_V.axvline(0, c='k', lw=5)
ax_I.axvline(0, c='k', lw=5)
ax_V.text(d-2, 0.4, 'input')
ax_V.text(1, 0.6, 'load')
ax_V.axvline(d, c='k', lw=5)
ax_I.axvline(d, c='k', lw=5)

ax_I.set_title('Current')
ax_V.set_title('Voltage')

## 使用 `media` 对象进行传输线计算

`scikit-rf` 还提供了用于表示传输线介质的对象。`Media` 对象提供了通用的方法，用于为任何传输线介质生成网络对象，例如传输线长度 (`line()`)、集总元件 (`resistor()`, `capacitor()`, `inductor()`, `shunt()` 等) 或端接元件 (`open()`, `short()`, `load()`)。有关更多参考信息，请参阅 [介质文档](../../api/media/index.rst)。现在，让我们创建一个传输线 `media` 对象，用于表示我们的同轴电缆，其特性阻抗为 $Z_0$，传播常数为 $\gamma$。

In [ ]:
# if not passing the gamma parameter, it would assume that gamma = alpha + j*beta = 0 + j*1
coax_line = rf.media.DefinedGammaZ0(frequency=freq, z0=Z_0, gamma=gamma)

为了构建如图所示的电路，首先创建电路中的所有网络，然后使用 `**` 运算符将它们[级联](../../tutorials/Networks.rst#Cascading-and-De-embedding)：<img src="transmission_line_properties_networks.svg">* 长度为 $d$ 的[传输线](../../api/media/generated/skrf.media.Media.line.rst)（来自上述创建的介质），* 阻抗为 $Z_L$ 的[电阻器](../../api/media/generated/skrf.media.Media.resistor.rst)，* 然后连接一个[短路](../../api/media/generated/skrf.media.Media.short.rst)。这将产生一个单端口网络，其 $Z$ 参数即为输入阻抗：

In [ ]:
ntw = coax_line.line(d, unit='m') ** coax_line.resistor(Z_L) ** coax_line.short()
ntw.z

请注意，可以使用便捷函数 [load](../../api/media/generated/skrf.media.Media.load.rst) 构建完整的网络模型。

In [ ]:
ntw = coax_line.line(d, unit='m') ** coax_line.load(zl_2_Gamma0(Z_0, Z_L))
ntw.z

或者更直接地使用 `delay_load`：

In [ ]:
ntw = coax_line.delay_load(zl_2_Gamma0(Z_0, Z_L), d, unit='m')
ntw.z

## 从输入阻抗确定传播常数 <a class="anchor" id="propagation_constant_from_zin"></a>假设我们测量了一条长度为 d=1.5 米，特性阻抗为 $Z_0$=100 欧姆的短路损耗传输线的输入阻抗，结果为 $Z_{in}=40 - 280j \Omega$。<img src="transmission_line_properties_propagation_constant.svg">传输线的传播常数 $\gamma$ 是未知的，我们需要研究如何使用 `scikit-rf` 推导出其值。

In [ ]:
# input data
z_in = 20 - 140j
z_0 = 75
d = 1.5
Gamma_load = -1 # short

由于我们已知输入阻抗，因此可以推导出传输线输入端的反射系数。由于负载端和线路输入端的反射系数之间存在方向关系：$$\Gamma_{in} = \Gamma_L e^{- 2 \gamma d}$$我们可以推导出传播常数 $\gamma$ 的值：$$\gamma = -\frac{1}{2d} \ln \left( \frac{\Gamma_{in}}{\Gamma_l} \right)$$`reflection_coefficient_2_propagation_constant` 函数的作用就是执行上述计算。

In [ ]:
# reflection coefficient at input
Gamma_in = zl_2_Gamma0(z_0, z_in)
# line propagation constant
gamma = reflection_coefficient_2_propagation_constant(Gamma_in, Gamma_load, d)
print('Line propagation constant, gamma =', gamma, 'rad/m')

可以通过进行反向计算来检查结果的一致性：计算距离负载 $Z_l$ 处 $d$ 距离处的输入阻抗：

In [ ]:
zl_2_zin(z_0, zl=0, theta=gamma * d)

这实际上就是示例中给定的输入值。

现在已经确定了线路的传播常数，就可以用负载电阻来代替短路：

In [ ]:
zl_2_zin(z_0, zl=50+50j, theta=gamma * d)